# Segment evaluation harness

Implements §4 of `docs/model-evaluation-strategy.md`. That document is the spec;
this notebook is the thing that runs it. The section numbers below match it.

| This notebook | Strategy §4 |
|---|---|
| §2 Baselines | 4.1 — a model that does not beat them is not worth its cost |
| §3 Per class, minority tiers | 4.2 — no score without its support count |
| §4 Calibration | 4.3 — reliability, ECE, Brier, temperature scaling |
| §5 Selective prediction | 4.4 — coverage against accuracy on answered rows |
| §6 Review queue | 4.5 — five routing rules, flag precision, miss rate, load |
| §7 Model comparison | 4.6 — paired, with uncertainty, cost beside accuracy |
| §8 Error inspection | where the models disagree, and on what |

**Reusable for the other five targets.** Nothing is hardcoded to Segment. Set
`TARGET` in §0 and everything follows: the label column, the baselines, the class
tiers, the per-class tables. The one thing that will not move is that the
transformer's own tokenizer reads a `Segment` column, so another target needs the
team's trainer pointed at it before there are predictions to evaluate here.

**It never reads the reserved test.** Predictions are validation-only by
construction, and §0 asserts that no `record_id` in any predictions file belongs
to the test partition.

## What it needs

One or more prediction directories written by the team's trainer, each holding:

```text
predictions.csv        record_id, true_label, predicted_label, confidence, probability_0..n-1
training_manifest.json label_names, in the order the probability columns use
```

`data/processed/` is gitignored, so those directories are not in the repo. If
none are present, §1 builds a reference baseline itself from TF-IDF and logistic
regression and writes it in the same contract — so the harness runs, and the
baseline it produces is the one §4.1 of the strategy asks for anyway.

## 0. Setup

The split directory is located by `protocol_manifest.json` and the candidate file
by its documented byte hash, the same guard as the research notebook. A V3
directory is refused.

In [ ]:
import hashlib, json, math, re, sys, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, brier_score_loss)

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 300)
print("pandas", pd.__version__, " numpy", np.__version__)

# ---- which target -------------------------------------------------------
TARGET = "Segment"          # change this and the whole notebook follows
SUPPORT_FLOOR = 200        # classes below this are reported in their own tier
RANDOM_STATE = 0

def project_root(start="."):
    p = Path(start).resolve()
    for _ in range(8):
        if (p/"data"/"provided").is_dir() or (p/".git").exists(): return p
        if p.parent == p: break
        p = p.parent
    return Path.cwd()

ROOT = project_root()
if (ROOT/"src").is_dir(): sys.path.insert(0, str(ROOT/"src"))
DATA, PROC = ROOT/"data", ROOT/"data"/"processed"
print("project root:", ROOT)
print("target      :", TARGET)

EXPECTED_CANDIDATE = "1878e3948e8a840b3922780b7304e33d39d0f1e3b5d7a579bf56639d3cdf8fb4"

def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for b in iter(lambda: fh.read(chunk), b""): h.update(b)
    return h.hexdigest()

cands = sorted(PROC.glob("*/training_candidate.csv"), key=lambda p: p.stat().st_mtime)
if not cands: raise SystemExit(f"No training_candidate.csv under {PROC}. Run the cleaner.")
match = [p for p in cands if sha256(p) == EXPECTED_CANDIDATE]
CANDIDATE = match[-1] if match else cands[-1]
print("candidate   :", CANDIDATE.relative_to(ROOT),
      "(hash matches)" if match else "(!! hash does NOT match the documented value)")

finalized = [p.parent for p in PROC.glob("*/rule_assignments.csv")
             if (p.parent/"protocol_manifest.json").is_file()]
if not finalized:
    raise SystemExit("No finalized split directory (needs protocol_manifest.json). "
                     "Run: python -m scripts.split_data --input <candidate> --output-dir <new dir> "
                     "and then --verify it.")
SPLIT_DIR = sorted(finalized, key=lambda p: p.stat().st_mtime)[-1]
print("split dir   :", SPLIT_DIR.relative_to(ROOT))
PROTOCOL_ID = (json.loads((SPLIT_DIR/"protocol_manifest.json").read_text())
                   .get("protocol_id", "unknown"))
print("protocol    :", PROTOCOL_ID)

RD = dict(dtype="string", keep_default_na=False, low_memory=False)
KEY = ["dataset", "source_sha256", "source_row"]
cand = pd.read_csv(CANDIDATE, **RD)
asg = pd.read_csv(SPLIT_DIR/"rule_assignments.csv", **RD)
frame = cand.merge(asg[KEY+["split", "group_id", "record_id"]], on=KEY,
                   how="inner", validate="one_to_one")
blank = lambda s: s.str.strip().str.lower().isin(["", "null"])
frame = frame[~blank(frame[TARGET])].copy()
frame["y"] = frame[TARGET].str.strip()

TEST_IDS = set(frame.loc[frame["split"] == "test", "record_id"])
TR = frame[frame["split"] == "train"]
VA = frame[frame["split"] == "validation"]
print(f"\nrows with a {TARGET} label: {len(frame):,}")
print(f"  train {len(TR):,}   validation {len(VA):,}   test {len(TEST_IDS):,} (reserved)")
print(f"  classes: {frame['y'].nunique()}")

### 0.1 Loading predictions

Each prediction directory is read against the contract the team's trainer writes.
Three things are asserted rather than assumed: the probability columns match
`label_names`, the rows are validation rows, and **no row belongs to the reserved
test**.

In [ ]:
VA_IDS = set(VA["record_id"])
GROUP_OF = dict(zip(frame["record_id"], frame["group_id"]))

# The trainer records validation_record_ids_sha256 so a run can be tied to the
# split it was trained against. Recompute it here from our own split and check
# any run that carries it. The recipe is _hash_items() in segment_transformer:
# ids in order, each prefixed with its UTF-8 byte length as 8 bytes big-endian.
def hash_items(items):
    d = hashlib.sha256()
    for item in items:
        payload = str(item).encode("utf-8")
        d.update(len(payload).to_bytes(8, "big"))
        d.update(payload)
    return d.hexdigest()

# Two conventions are accepted: over the labelled validation rows (what the
# trainer uses) and over the whole validation partition, in case a teammate
# hashed before dropping unlabelled rows. Anything else is a different split.
VA_ID_HASHES = {
    "labelled validation rows": hash_items(sorted(VA_IDS)),
    "whole validation partition": hash_items(
        sorted(asg.loc[asg["split"] == "validation", "record_id"])),
}
print("validation record-id hash (labelled rows):", VA_ID_HASHES["labelled validation rows"])

# Teammates package runs differently: the trainer writes training_manifest.json,
# the baseline handoff writes label_mapping.json and manifest.json. Read all of
# them rather than making one packaging convention the only one that loads.
MANIFESTS = ("training_manifest.json", "protocol.json", "model_config.json",
             "label_mapping.json", "manifest.json")

def _labels_from(obj):
    # Pull an ordered class list out of any of the shapes these files use.
    if isinstance(obj, list):
        return [str(x) for x in obj] or None
    if not isinstance(obj, dict): return None
    for k in ("label_names", "labels", "classes", "class_names", "label_mapping"):
        v = obj.get(k)
        if isinstance(v, list) and v: return [str(x) for x in v]
        if isinstance(v, dict) and v: obj = v; break
    else:
        if not all(isinstance(x, (int, str)) for x in obj.values()): return None
    # index -> class, or class -> index; decide by which side holds the indices.
    # An index may be bare (3) or carried by a column name ("probability_3"),
    # which is the shape the baseline handoff writes.
    def idx(x):
        if isinstance(x, bool): return None
        if isinstance(x, int): return x
        m = re.fullmatch(r"\D*?(\d+)", str(x).strip())
        return int(m.group(1)) if m else None
    def positions(xs):
        got = [idx(x) for x in xs]
        return got if None not in got and sorted(got) == list(range(len(got))) else None
    keys, vals = list(obj.keys()), list(obj.values())
    pos = positions(keys)
    if pos: return [str(v) for _, v in sorted(zip(pos, obj.values()))]
    pos = positions(vals)
    if pos: return [str(k) for _, k in sorted(zip(pos, obj.keys()))]
    return None

def load_predictions(directory, name=None):
    directory = Path(directory)
    name = name or directory.name
    p = pd.read_csv(directory/"predictions.csv",
                    dtype={"record_id": str, "true_label": str, "predicted_label": str})
    need = {"record_id", "true_label", "predicted_label", "confidence"}
    missing = need - set(p.columns)
    if missing: raise ValueError(f"{name}: predictions.csv missing {sorted(missing)}")

    labels, claimed = None, {}
    for fn in MANIFESTS:
        f = directory/fn
        if not f.is_file(): continue
        obj = json.loads(f.read_text())
        if isinstance(obj, dict) and obj.get("validation_record_ids_sha256"):
            claimed[fn] = obj["validation_record_ids_sha256"]
        if labels is None:
            labels = _labels_from(obj) or None

    # A run that names the split it trained on gets checked against ours.
    for fn, digest in claimed.items():
        hit = [k for k, v in VA_ID_HASHES.items() if v == digest]
        if not hit:
            raise ValueError(
                f"{name}: {fn} claims validation_record_ids_sha256 {digest[:16]}... "
                f"which matches neither this split's labelled validation rows nor its "
                f"whole validation partition. The run was trained against a different "
                f"split, so refusing to score it.")
        print(f"  {name:28} validation ids match this split ({hit[0]})")
    prob_cols = sorted((c for c in p.columns if c.startswith("probability_")),
                       key=lambda c: int(c.split("_")[1]))
    if labels is None and prob_cols:
        raise ValueError(f"{name}: has probability columns but no class order in any of "
                         f"{list(MANIFESTS)}, so they cannot be mapped to classes")
    if prob_cols and len(prob_cols) != len(labels):
        raise ValueError(f"{name}: {len(prob_cols)} probability columns against "
                         f"{len(labels)} classes")
    # Do not take the class order on trust. If probability_i is not class i, the
    # argmax stops agreeing with predicted_label, and every calibration and
    # margin number below is computed on the wrong column. Check it against the
    # data instead of believing the manifest.
    if prob_cols:
        argmax = np.asarray(labels)[p[prob_cols].to_numpy(dtype=float).argmax(axis=1)]
        off = (argmax != p["predicted_label"].str.strip().to_numpy()).mean()
        if off > 0.01:
            raise ValueError(
                f"{name}: the argmax of the probability columns disagrees with "
                f"predicted_label on {off*100:.1f}% of rows. The class order taken "
                f"from the manifest is probably wrong, so refusing to score it.")
        if off:
            print(f"  !! {name}: argmax disagrees with predicted_label on {off*100:.2f}% "
                  f"of rows (ties, or a stale predicted_label column)")

    leaked = set(p["record_id"]) & TEST_IDS
    if leaked:
        raise ValueError(f"{name}: {len(leaked)} prediction rows belong to the RESERVED "
                         f"TEST partition. Refusing to score them.")
    outside = set(p["record_id"]) - VA_IDS
    if outside:
        print(f"  !! {name}: {len(outside):,} rows are not validation rows; dropping them")
        p = p[p["record_id"].isin(VA_IDS)]

    p["group_id"] = p["record_id"].map(GROUP_OF)
    p["correct"] = p["true_label"].str.strip() == p["predicted_label"].str.strip()
    if prob_cols:
        P = p[prob_cols].to_numpy(dtype=float)
        order = np.argsort(-P, axis=1)
        p["top1_prob"] = P[np.arange(len(P)), order[:, 0]]
        p["top2_prob"] = P[np.arange(len(P)), order[:, 1]] if P.shape[1] > 1 else 0.0
        p["margin"] = p["top1_prob"] - p["top2_prob"]
        p["_probs"] = list(P)
    else:
        p["top1_prob"] = pd.to_numeric(p["confidence"], errors="coerce")
        p["margin"] = np.nan
        p["_probs"] = None
    print(f"  {name:28} {len(p):,} rows   accuracy {p['correct'].mean()*100:6.2f}%   "
          f"{'with' if prob_cols else 'no'} probabilities")
    return {"name": name, "frame": p, "labels": labels, "dir": directory,
            "split_checked": bool(claimed)}

def _check_label_reader():
    # The class order decides every number below, so prove the reader, don't assume it.
    C = ["a", "b", "c"]
    shapes = [{"label_names": C}, C, {str(i): c for i, c in enumerate(C)},
              {i: c for i, c in enumerate(C)}, {c: i for i, c in enumerate(C)},
              {"label_mapping": {str(i): c for i, c in enumerate(C)}}, {"classes": C},
              {"2": "c", "0": "a", "1": "b"},
              # the shape scripts.package_segment_baseline writes
              {f"probability_{i}": c for i, c in enumerate(C)},
              {"label_mapping": {f"probability_{i}": c for i, c in enumerate(C)}}]
    for s in shapes:
        assert _labels_from(s) == C, f"class order misread from {s}"
    for s in ({"protocol_id": "x", "rows": 1}, {"a": "x", "b": "y"},
              {"probability_0": "a", "probability_2": "b"}):     # gap in the indices
        assert _labels_from(s) is None,             f"a file with no usable class order must return None, not a guess: {s}"
_check_label_reader()

RUNS = {}
# The harness writes its own reference baselines under this directory in §1.
# Auto-discovery must skip them, or they load twice and §7 compares a run
# against itself.
OWN_BASELINES = PROC/"interim_eval_baseline"

print("prediction directories found (team runs only):")
for d in sorted(PROC.glob("**/predictions.csv")):
    if OWN_BASELINES in d.parents: continue
    if (any((d.parent/m).is_file() for m in MANIFESTS)
            or (d.parent.parent/"protocol.json").is_file()):
        key = f"{d.parent.parent.name}/{d.parent.name}" if d.parent.parent != PROC else d.parent.name
        try:
            RUNS[key] = load_predictions(d.parent, key)
        except Exception as e:
            print(f"  skipped {key}: {type(e).__name__}: {e}")
if not RUNS:
    print("  none. §1 will build a reference baseline in the same contract.")

## 1. Two reference baselines, in the team's own contract

§4.1 of the strategy asks for a TF-IDF plus linear classifier baseline. Building
it here does several jobs at once: it gives the harness something real to run on
without a GPU, it is the baseline any model has to beat, and it produces the
probability columns §4.3 and §4.4 need.

**Two arms, not one**, because the ticket asks to compare results *before and
after a feature change*. The arms differ only in whether `ProductCategory` is in
the text, which is the change the research recommended. That also means §7 has
two runs to compare and its paired statistics actually execute rather than
sitting untested until a checkpoint shows up.

Logistic regression rather than a linear SVM, because §4.3 needs probabilities.

In [ ]:
FIVE = ["ProductCategory", "ProductName", "ProductBrand",
        "ProductDescription", "ProductContents"]
FOUR = ["ProductName", "ProductBrand", "ProductDescription", "ProductContents"]

def build_text(df, fields):
    parts = []
    for c in fields:
        if c not in df.columns: continue
        v = df[c].astype("string").fillna("").str.strip().replace({"null": ""})
        parts.append(("[" + c + "] " + v).where(v != "", ""))
    return pd.concat(parts, axis=1).agg(" ".join, axis=1).str.replace(r"\s+", " ", regex=True)

def write_arm(tag, fields):
    d = PROC/"interim_eval_baseline"/f"tfidf_logreg_{TARGET.lower()}_{tag}"
    if (d/"predictions.csv").is_file():
        print(f"  {tag}: already present")
        return d
    labels = sorted(frame["y"].unique())
    clf = make_pipeline(
        TfidfVectorizer(max_features=80000, ngram_range=(1, 2), min_df=2, sublinear_tf=True),
        LogisticRegression(max_iter=2000, class_weight="balanced",
                           random_state=RANDOM_STATE, n_jobs=-1))
    clf.fit(build_text(TR, fields), TR["y"])
    P = clf.predict_proba(build_text(VA, fields))
    cls = list(clf.classes_)
    P = P[:, [cls.index(l) for l in labels]]          # reorder to `labels`
    pred = pd.DataFrame({
        "record_id": VA["record_id"].to_numpy(),
        "true_label": VA["y"].to_numpy(),
        "predicted_label": [labels[i] for i in P.argmax(axis=1)],
        "confidence": P.max(axis=1)})
    for i in range(len(labels)): pred[f"probability_{i}"] = P[:, i]
    d.mkdir(parents=True, exist_ok=True)
    pred.to_csv(d/"predictions.csv", index=False)
    (d/"training_manifest.json").write_text(json.dumps({
        "label_names": labels, "model": "tfidf+logreg", "target": TARGET,
        "input_fields": [c for c in fields if c in frame.columns],
        # Declare the split, same as the trainer does, so these arms are held to
        # the same check as anyone else's run rather than exempt from it.
        "protocol_id": PROTOCOL_ID,
        "validation_record_ids_sha256": VA_ID_HASHES["labelled validation rows"],
        "note": "Reference baseline written by the evaluation harness, in the team's "
                "predictions contract. Not a team checkpoint."}, indent=2))
    print(f"  {tag}: written to {d.relative_to(ROOT)}")
    return d

print("reference baselines (TF-IDF + logistic regression):")
ARMS = {"baseline_4field": write_arm("4field", FOUR),
        "baseline_5field": write_arm("5field", FIVE)}
for name, d in ARMS.items():
    RUNS[name] = load_predictions(d, name)
print(f"\nruns available for evaluation: {list(RUNS)}")

## 2. Baselines (§4.1)

Three floors. A model that does not clearly clear all three is not worth its
operating cost, which is the strategy's wording and the right standard.

In [ ]:
vc = TR["y"].value_counts()
majority = VA["y"].value_counts().iloc[0]/len(VA)
print(f"majority class        : {majority*100:6.2f}% accuracy   "
      f"({VA['y'].value_counts().index[0]})")
print(f"{'':24}  macro F1 is {f1_score(VA['y'], [vc.index[0]]*len(VA), average='macro')*100:.2f}% "
      f"- the floor on the metric that matters")

LOOKUP_COL = {"Segment": "ProductCategory", "Sub-Segment": "ProductCategory",
              "Brand": "ProductBrand", "Mnfr": "Brand",
              "Platform": "ProductCategory", "TargetAgeGroup": "ProductName"}.get(TARGET)
if LOOKUP_COL and LOOKUP_COL in frame.columns:
    norm = lambda s: s.astype("string").str.strip()
    table = TR.assign(k=norm(TR[LOOKUP_COL])).query("k != ''") \
              .groupby("k")["y"].agg(lambda g: g.value_counts().index[0])
    k = norm(VA[LOOKUP_COL])
    hit = k.isin(table.index)
    guess = k.map(table).fillna(vc.index[0])
    print(f"lookup on {LOOKUP_COL:<12}: {accuracy_score(VA['y'], guess)*100:6.2f}% accuracy   "
          f"macro F1 {f1_score(VA['y'], guess, average='macro')*100:6.2f}%")
    print(f"{'':24}  covers {hit.mean()*100:.1f}% of validation rows from "
          f"{len(table):,} keys learned on train")
else:
    print(f"lookup baseline       : no lookup column defined for {TARGET}")

ref = RUNS["baseline_5field"]["frame"]
print(f"TF-IDF + logreg       : {ref['correct'].mean()*100:6.2f}% accuracy   "
      f"macro F1 {f1_score(ref['true_label'], ref['predicted_label'], average='macro')*100:6.2f}%")

print()
rows = []
for name, run in RUNS.items():
    f = run["frame"]
    rows.append({"run": name, "n": len(f),
                 "accuracy_%": round(f["correct"].mean()*100, 2),
                 "macro_F1_%": round(f1_score(f["true_label"], f["predicted_label"],
                                              average="macro")*100, 2)})
SCORES = pd.DataFrame(rows).sort_values("macro_F1_%", ascending=False)
print(SCORES.to_string(index=False))
print()
print("A run below the lookup or TF-IDF line above is not worth its cost, however")
print("good its headline reads.")

## 3. Per class, with minority classes in their own tier (§4.2)

The strategy is explicit: no score is reported without its support count, and
classes below the floor go in their own tier rather than being averaged into the
headline. A macro average over a class with 170 rows and a class with 7,000
hides exactly the thing the project cares about.

In [ ]:
def per_class(run):
    f = run["frame"]
    rep = classification_report(f["true_label"], f["predicted_label"],
                                output_dict=True, zero_division=0)
    rows = []
    for k, v in rep.items():
        if k in ("accuracy", "macro avg", "weighted avg"): continue
        rows.append({"class": k, "support": int(v["support"]),
                     "precision": round(v["precision"], 3),
                     "recall": round(v["recall"], 3),
                     "f1": round(v["f1-score"], 3)})
    t = pd.DataFrame(rows).sort_values("support", ascending=False)
    t["tier"] = np.where(t["support"] >= SUPPORT_FLOOR, "main", "minority")
    return t

for name, run in RUNS.items():
    t = per_class(run)
    print(f"=== {name} ===")
    print(t.to_string(index=False))
    # Average the unrounded per-class scores. Averaging the display-rounded
    # column instead disagrees with §2 and §9 by up to 0.01pp, which reads as
    # two different numbers for the same quantity.
    fr = run["frame"]
    exact = {l: v["f1-score"] for l, v in classification_report(
        fr["true_label"], fr["predicted_label"], labels=run["labels"],
        output_dict=True, zero_division=0).items() if l in run["labels"]}
    for tier in ("main", "minority"):
        sub = t[t["tier"] == tier]
        if len(sub):
            print(f"  {tier:8} classes {len(sub):2}  support {sub['support'].sum():>6,}  "
                  f"macro F1 {np.mean([exact[c] for c in sub['class']])*100:6.2f}%")
    print(f"  headline macro F1 over all classes: "
          f"{np.mean(list(exact.values()))*100:.2f}%  <- report the two tiers above beside it, never alone")
    print()

## 4. Calibration (§4.3)

The README makes confidence a product requirement, so the confidence has to mean
something. A threshold of 0.80 is meaningless unless rows at 0.80 are right about
80% of the time.

Reported: a reliability table, expected calibration error, and the Brier score.
Then temperature scaling, which changes the confidence and never the predicted
class — so it cannot cost accuracy.

In [ ]:
def ece_and_reliability(conf, correct, bins=10):
    conf = np.asarray(conf, dtype=float); correct = np.asarray(correct, dtype=float)
    edges = np.linspace(0, 1, bins+1)
    rows, ece = [], 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi) if lo > 0 else (conf >= lo) & (conf <= hi)
        if not m.any(): continue
        acc, mean_conf, share = correct[m].mean(), conf[m].mean(), m.mean()
        ece += share*abs(acc-mean_conf)
        rows.append({"bin": f"{lo:.1f}-{hi:.1f}", "rows": int(m.sum()),
                     "share_%": round(share*100, 1),
                     "mean_confidence": round(mean_conf, 3),
                     "accuracy": round(acc, 3),
                     "gap": round(acc-mean_conf, 3)})
    return pd.DataFrame(rows), ece

def multiclass_brier(probs, true_label, labels):
    if probs is None: return float("nan")
    P = np.vstack(probs)
    idx = {l: i for i, l in enumerate(labels)}
    Y = np.zeros_like(P)
    for r, t in enumerate(true_label):
        j = idx.get(str(t).strip())
        if j is not None: Y[r, j] = 1.0
    return float(((P-Y)**2).sum(axis=1).mean())

def temperature_scale(probs, true_label, labels, grid=None):
    """One parameter on the logits. Argmax is invariant, so accuracy cannot move."""
    P = np.clip(np.vstack(probs), 1e-12, 1.0)
    logits = np.log(P)
    idx = {l: i for i, l in enumerate(labels)}
    y = np.array([idx.get(str(t).strip(), -1) for t in true_label])
    keep = y >= 0
    logits, y = logits[keep], y[keep]
    best = (1.0, float("inf"))
    for T in (grid if grid is not None else np.linspace(0.5, 5.0, 46)):
        z = logits/T
        z = z - z.max(axis=1, keepdims=True)
        nll = -(z[np.arange(len(y)), y] - np.log(np.exp(z).sum(axis=1))).mean()
        if nll < best[1]: best = (float(T), float(nll))
    T = best[0]
    z = logits/T; z = z - z.max(axis=1, keepdims=True)
    Q = np.exp(z); Q = Q/Q.sum(axis=1, keepdims=True)
    return T, Q, keep

CAL = {}
for name, run in RUNS.items():
    f, labels = run["frame"], run["labels"]
    print(f"=== {name} ===")
    rel, ece = ece_and_reliability(f["top1_prob"], f["correct"])
    print(rel.to_string(index=False))
    brier = multiclass_brier(f["_probs"].tolist() if f["_probs"] is not None else None,
                             f["true_label"], labels) if labels else float("nan")
    print(f"  expected calibration error : {ece*100:.2f}%")
    print(f"  Brier score (multiclass)   : {brier:.4f}" if brier == brier else
          "  Brier score                : not available without probabilities")
    if labels and f["_probs"] is not None:
        T, Q, keep = temperature_scale(f["_probs"].tolist(), f["true_label"], labels)
        newconf = Q.max(axis=1)
        _, ece2 = ece_and_reliability(newconf, f.loc[keep, "correct"])
        print(f"  temperature                : T = {T:.2f}")
        print(f"  ECE after scaling          : {ece2*100:.2f}%  "
              f"({'better' if ece2 < ece else 'no better'})")
        acc_before = f.loc[keep, 'correct'].mean()
        acc_after = (np.array([labels[i] for i in Q.argmax(axis=1)]) ==
                     f.loc[keep, 'true_label'].str.strip().to_numpy()).mean()
        print(f"  accuracy before / after    : {acc_before*100:.2f}% / {acc_after*100:.2f}%")
        # Temperature scaling divides the logits by a positive scalar, so the
        # argmax cannot move and accuracy cannot change. Checked, not trusted:
        # if this ever fires, the scaling implementation is wrong.
        assert abs(acc_before - acc_after) < 1e-9, (
            f"temperature scaling changed accuracy ({acc_before:.6f} -> {acc_after:.6f}); "
            f"the predicted class must be invariant under a positive temperature")
        print(f"  checked: scaling left the predicted class unchanged on every row")
        CAL[name] = {"T": T, "ece": ece, "ece_scaled": ece2, "brier": brier,
                     "calibrated_conf": newconf, "keep": keep}
    print()

## 5. Selective prediction (§4.4)

"70% accurate on everything" and "95% accurate on the 60% it is sure about" are
different products, and only the second lets the PO choose an operating point.
Abstention is not an error and is counted separately.

Coverage is set by taking the most confident rows first.

In [ ]:
def coverage_risk(f, labels, conf_col="top1_prob", points=(1.00, 0.90, 0.75, 0.50, 0.25)):
    s = f.sort_values(conf_col, ascending=False)
    rows = []
    for c in points:
        k = max(1, int(round(len(s)*c)))
        head = s.iloc[:k]
        # Taking the most confident rows first does not keep the class mix. On a
        # confident model the top half is almost all majority class, so a macro
        # average over whatever survives is not the same quantity as the macro F1
        # at full coverage and must not be put in the same column. Report it only
        # while every class is still represented.
        kept = sorted(set(head["true_label"]) | set(head["predicted_label"]))
        full = len(kept) == len(labels)
        rows.append({"coverage_%": round(c*100, 1), "answered": k,
                     "abstained": len(s)-k,
                     "accuracy_on_answered_%": round(head["correct"].mean()*100, 2),
                     "classes_left": len(kept),
                     "macro_F1_on_answered_%": (round(f1_score(head["true_label"],
                         head["predicted_label"], average="macro",
                         labels=labels, zero_division=0)*100, 2) if full else np.nan),
                     "confidence_threshold": round(float(head[conf_col].min()), 4)})
    return pd.DataFrame(rows)

for name, run in RUNS.items():
    t = coverage_risk(run["frame"], run["labels"])
    print(f"=== {name} ===")
    print(t.to_string(index=False, na_rep="n/a"))
    lost = t[t["classes_left"] < len(run["labels"])]
    if len(lost):
        print(f"  macro F1 is n/a below {lost['coverage_%'].max():.0f}% coverage: only "
              f"{lost['classes_left'].max()} of {len(run['labels'])} classes survive the "
              f"confidence cut, so the average is over a different class set.")
        print("  accuracy on the answered rows is still comparable; the macro average is not.")
    print()
print("Read the threshold column: that is the cutoff the product would ship at to")
print("buy the accuracy on its row. Use the calibrated confidence from §4 for a")
print("threshold anyone outside the team is expected to interpret.")

### 5.1 Confidence against margin, at the same review load

§4.5 of the strategy guesses that margin "often catches more real errors than
raw confidence". Comparing the two rules at their own thresholds cannot answer
that, because a threshold on one is not the same amount of work as a threshold
on the other. Equal flag precision at twice the review load is a worse rule, not
a tie. So fix the load and ask which rule finds more errors inside it.

In [ ]:
def rule_race(run, loads=(0.01, 0.02, 0.05, 0.10, 0.15, 0.25)):
    f = run["frame"]
    if f["margin"].isna().all(): return None
    rows = []
    for L in loads:
        k = max(1, int(round(len(f)*L)))
        got = {}
        for col in ("top1_prob", "margin"):
            sel = f.nsmallest(k, col)
            got[col] = (~sel["correct"]).sum()/k*100
        rows.append({"review_load_%": round(L*100, 1), "flagged": k,
                     "confidence_precision_%": round(got["top1_prob"], 2),
                     "margin_precision_%": round(got["margin"], 2),
                     "margin_minus_confidence": round(got["margin"]-got["top1_prob"], 2)})
    return pd.DataFrame(rows)

for name, run in RUNS.items():
    t = rule_race(run)
    if t is None: continue
    print(f"=== {name} ===")
    print(t.to_string(index=False))
    d = t["margin_minus_confidence"]
    wins = (d > 0.5).sum(); losses = (d < -0.5).sum()
    print(f"  margin ahead by >0.5pp at {wins} of {len(d)} loads, behind at {losses}, "
          f"within 0.5pp at {len(d)-wins-losses}")
    print(f"  largest gap either way: {d.abs().max():.2f}pp\n")
print("A rule that only wins at one review load has not beaten the other rule;")
print("it has found one operating point. The answer differs per model, so this")
print("is not a project-wide choice.")

## 6. The review queue (§4.5)

Five routing rules from the strategy, scored the way it asks — flag precision,
miss rate and review load, across thresholds rather than at one cutoff. A queue
with no metrics is an unmeasured cost.

Rule 4 (predicted placeholder) and rule 5 (relationship violations) need the
placeholder vocabulary and the R1–R6 rules from the relationships research; they
are wired in where the inputs are present and skipped, loudly, where they are
not.

In [ ]:
# Rule 4 in the strategy is keyed to a target's own placeholder classes - the
# example given is "UnItemised brand" for a row whose ProductBrand is populated.
# It is a Brand/Mnfr rule. Segment has no placeholder class: `Other Self Care`
# is a real class with 173 validation rows, and flagging it here would just be
# rule 3 again under another name.
PLACEHOLDER_CLASSES = {
    "Brand": ({"unitemised brand", "unitemized brand", "unknown brand"}, "ProductBrand"),
    "Mnfr":  ({"all others", "unknown"}, "Brand"),
}
PLACEHOLDERS, PLACEHOLDER_SOURCE = PLACEHOLDER_CLASSES.get(TARGET, (set(), None))

def queue_metrics(flagged, correct):
    flagged = np.asarray(flagged, dtype=bool); wrong = ~np.asarray(correct, dtype=bool)
    n = len(flagged)
    return {"review_load_%": round(flagged.mean()*100, 2),
            "flag_precision_%": round((wrong & flagged).sum()/max(flagged.sum(), 1)*100, 2),
            "miss_rate_%": round((wrong & ~flagged).sum()/max(wrong.sum(), 1)*100, 2),
            "wrong_caught": int((wrong & flagged).sum()), "wrong_total": int(wrong.sum())}

for name, run in RUNS.items():
    f, labels = run["frame"], run["labels"]
    conf = CAL[name]["calibrated_conf"] if name in CAL else None
    print(f"=== {name} ===")

    print("rule 1 - low top-1 confidence" + (" (calibrated)" if conf is not None else ""))
    use = pd.Series(conf, index=f.index) if conf is not None else f["top1_prob"]
    rows = [dict(threshold=t, **queue_metrics(use < t, f["correct"]))
            for t in (0.50, 0.70, 0.80, 0.90, 0.95)]
    print(pd.DataFrame(rows).to_string(index=False))

    if f["margin"].notna().any():
        print("\nrule 2 - small margin between top-1 and top-2")
        rows = [dict(threshold=t, **queue_metrics(f["margin"] < t, f["correct"]))
                for t in (0.05, 0.10, 0.20, 0.40, 0.60)]
        print(pd.DataFrame(rows).to_string(index=False))

    print("\nrule 3 - rare predicted class (training support below the floor)")
    rare = {c for c, n in TR["y"].value_counts().items() if n < SUPPORT_FLOOR*5}
    print(f"  classes treated as rare: {sorted(rare) if rare else 'none under the cutoff'}")
    m = queue_metrics(f["predicted_label"].str.strip().isin(rare), f["correct"])
    print("  " + "  ".join(f"{k} {v}" for k, v in m.items()))

    print("\nrule 4 - predicted a placeholder class while the source field is populated")
    if not PLACEHOLDERS:
        print(f"  {TARGET} has no placeholder class, so this rule does not apply to it.")
        print(f"  It is a Brand/Mnfr rule. `Other Self Care` is a real {TARGET} class and")
        print(f"  flagging it here would only repeat rule 3.")
    else:
        ph = f["predicted_label"].str.strip().str.lower().isin(PLACEHOLDERS)
        src = frame.set_index("record_id")[PLACEHOLDER_SOURCE].reindex(f["record_id"])
        populated = (~src.astype("string").fillna("").str.strip()
                       .str.lower().isin(["", "null"])).to_numpy()
        m = queue_metrics(ph.to_numpy() & populated, f["correct"])
        print(f"  placeholder predicted AND {PLACEHOLDER_SOURCE} populated")
        print("  " + "  ".join(f"{k} {v}" for k, v in m.items()))

    print("\nrule 5 - relationship violations R1-R6")
    print("  not wired: needs the rule definitions from the relationships research")
    print("  exported in a machine-readable form. Until then this rule is unmeasured,")
    print("  which is better than reporting it as passing.")

    if f["margin"].notna().any():
        print("\ncombined: low calibrated confidence OR small margin")
        base = use
        rows = []
        for t, mg in ((0.90, 0.20), (0.80, 0.10), (0.70, 0.05)):
            rows.append(dict(conf_below=t, margin_below=mg,
                             **queue_metrics((base < t) | (f["margin"] < mg), f["correct"])))
        print(pd.DataFrame(rows).to_string(index=False))
    print()

## 7. Comparing two models (§4.6)

Two requirements from the strategy, both easy to skip and both load-bearing:
**uncertainty on every comparison**, and **cost reported next to accuracy**.

McNemar for the paired disagreement, and a bootstrap resampled over **groups**
rather than rows, because rows inside a product group are not independent. A
several-point difference over a small class is usually noise, and this is what
says so.

In [ ]:
def mcnemar(a_correct, b_correct):
    a = np.asarray(a_correct, dtype=bool); b = np.asarray(b_correct, dtype=bool)
    n01 = int((~a & b).sum()); n10 = int((a & ~b).sum())
    if n01 + n10 == 0: return n01, n10, float("nan")
    chi2 = (abs(n01-n10)-1)**2/(n01+n10)          # continuity-corrected
    p = math.erfc(math.sqrt(chi2/2))              # 1 dof survival
    return n01, n10, p

def grouped_bootstrap(df, metric, samples=1000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    groups = df["group_id"].fillna(df["record_id"]).to_numpy()
    uniq, inv = np.unique(groups, return_inverse=True)
    by_group = [np.where(inv == i)[0] for i in range(len(uniq))]
    out = []
    for _ in range(samples):
        pick = rng.integers(0, len(uniq), len(uniq))
        idx = np.concatenate([by_group[i] for i in pick])
        out.append(metric(df.iloc[idx]))
    return np.percentile(out, [2.5, 97.5])

names = list(RUNS)
if len(names) < 2:
    print("Only one run loaded, so there is nothing to compare.")
    print("Point the harness at a second prediction directory - a team checkpoint, or")
    print("the four-field arm beside the five-field one - and this section fills in.")
else:
    MACRO = lambda d: f1_score(d["true_label"], d["predicted_label"], average="macro")
    ACC = lambda d: float(d["correct"].mean())
    for i in range(len(names)):
        for j in range(i+1, len(names)):
            A, B = RUNS[names[i]]["frame"], RUNS[names[j]]["frame"]
            common = sorted(set(A["record_id"]) & set(B["record_id"]))
            if not common:
                print(f"{names[i]} vs {names[j]}: no shared rows"); continue
            a = A.set_index("record_id").loc[common]
            b = B.set_index("record_id").loc[common]
            print(f"=== {names[i]}  vs  {names[j]}   on {len(common):,} shared rows ===")
            n01, n10, p = mcnemar(a["correct"], b["correct"])
            print(f"  {names[i]} right / {names[j]} wrong : {n10:,}")
            print(f"  {names[i]} wrong / {names[j]} right : {n01:,}")
            print(f"  McNemar p-value                     : "
                  + (f"{p:.2e}" if p == p else "n/a (no disagreement)"))
            for label, metric in (("accuracy", ACC), ("macro F1", MACRO)):
                da, db = metric(a.reset_index()), metric(b.reset_index())
                lo, hi = grouped_bootstrap(a.reset_index(), metric, samples=300)
                lo2, hi2 = grouped_bootstrap(b.reset_index(), metric, samples=300)
                overlap = not (hi < lo2 or hi2 < lo)
                verdict = ("intervals OVERLAP, so this difference is not established"
                           if overlap else "intervals are disjoint")
                print(f"  {label:9}: {names[i]} {da*100:.2f}% [{lo*100:.2f}, {hi*100:.2f}]   "
                      f"{names[j]} {db*100:.2f}% [{lo2*100:.2f}, {hi2*100:.2f}]   "
                      f"diff {(da-db)*100:+.2f}pp")
                print(f"  {'':9}  {verdict}")
            print("  95% group-aware bootstrap, resampled over product groups rather than")
            print("  rows. McNemar is the paired test on the same rows; the intervals say")
            print("  how much each score would move on a different sample of groups.")
            print()
    print("Cost belongs in this table and cannot be measured from predictions alone.")
    print("Fill it from each run's training_manifest.json: runtime_seconds, whether a")
    print("GPU was needed, and inference latency. A 2pp gain that triples cost is a")
    print("business decision, not an automatic win.")

## 8. Where the models disagree, and on what

Error inspection. The confusion pairs that dominate, and the rows where two runs
disagree — which is where a reviewer's time is best spent.

In [ ]:
for name, run in RUNS.items():
    f = run["frame"]
    wrong = f[~f["correct"]]
    pairs = (wrong.groupby([wrong["true_label"].str.strip(),
                            wrong["predicted_label"].str.strip()])
                  .size().sort_values(ascending=False).head(10))
    print(f"=== {name}: {len(wrong):,} errors, top confusions ===")
    for (t, p), n in pairs.items():
        print(f"  {n:>5}  {t}  ->  {p}")
    print(f"  the top 10 pairs are {pairs.sum()/max(len(wrong),1)*100:.1f}% of all errors")
    print()

if len(RUNS) >= 2:
    a_name, b_name = list(RUNS)[:2]
    A = RUNS[a_name]["frame"].set_index("record_id")
    B = RUNS[b_name]["frame"].set_index("record_id")
    common = sorted(set(A.index) & set(B.index))
    a, b = A.loc[common], B.loc[common]
    dis = a[(a["correct"] != b["correct"])]
    print(f"=== rows where {a_name} and {b_name} disagree on correctness: {len(dis):,} ===")
    joined = frame.set_index("record_id")
    show = dis.head(8)
    for rid, row in show.iterrows():
        src = joined.loc[rid] if rid in joined.index else None
        print(f"  {rid[:12]}  true={row['true_label']}")
        print(f"      {a_name}: {row['predicted_label']} (conf {row['top1_prob']:.3f})"
              f"   {b_name}: {b.loc[rid,'predicted_label']} (conf {b.loc[rid,'top1_prob']:.3f})")
        if src is not None:
            print(f"      name: {str(src.get('ProductName',''))[:88]}")
            print(f"      cat : {str(src.get('ProductCategory',''))[:88]}")
    print()
    print("These are the rows to read by hand before trusting either model, and the")
    print("rows a review rule should be catching.")

### 8.1 Does the source field agree with the label it was scored against?

When one confusion pair dominates, there are two possible causes and they need
different fixes. Either the model is missing something, or the label disagrees
with the strongest feature and no amount of training will close it.

This checks which, by reading the source field on every row of the dominant pair
and asking whether it names the true label, the predicted label, both, or
neither. The cue words come from the class names themselves, so it works for any
target; a class whose name carries no distinctive word is reported as
undecidable rather than silently counted as "neither".

In [ ]:
SOURCE_FIELD = {"Segment": "ProductCategory", "Sub-Segment": "ProductCategory",
                "Brand": "ProductBrand", "Mnfr": "Brand"}.get(TARGET)
GENERIC_WORDS = {"health", "household", "medicine", "other", "self", "care",
                 "shop", "products", "supply", "general"}

def class_cues(label, min_len=6):
    # Distinctive words of the class name, cut to a prefix so plurals and
    # inflections still match: "Digestive" -> "digest" also catches "digestion".
    words = [w for w in re.findall(r"[a-z]+", str(label).lower())
             if len(w) >= min_len and w not in GENERIC_WORDS]
    return {w[:min_len] for w in words}

def names_class(text, label):
    cues = class_cues(label)
    if not cues: return None                 # cannot be decided from the text
    low = str(text).lower()
    return any(c in low for c in cues)

if SOURCE_FIELD not in frame.columns:
    print(f"no source field for {TARGET}; skipping")
else:
    src = frame.set_index("record_id")
    for name, run in RUNS.items():
        f = run["frame"]; wrong = f[~f["correct"]]
        pairs = (wrong.groupby([wrong["true_label"].str.strip(),
                                wrong["predicted_label"].str.strip()]).size())
        if pairs.empty: continue
        A, B = pairs.idxmax()
        sel = wrong[wrong["true_label"].str.strip().isin([A, B])
                    & wrong["predicted_label"].str.strip().isin([A, B])]
        tally = {"names the PREDICTED label": 0, "names the TRUE label": 0,
                 "names neither": 0, "undecidable from the text": 0}
        for rid, row in sel.set_index("record_id").iterrows():
            text = src.loc[rid, SOURCE_FIELD] if rid in src.index else ""
            t = names_class(text, row["true_label"].strip())
            p_ = names_class(text, row["predicted_label"].strip())
            if t is None or p_ is None:   tally["undecidable from the text"] += 1
            elif p_:                      tally["names the PREDICTED label"] += 1
            elif t:                       tally["names the TRUE label"] += 1
            else:                         tally["names neither"] += 1
        print(f"=== {name}: dominant pair {A} <-> {B} ===")
        print(f"  {len(sel):,} of {len(wrong):,} errors ({len(sel)/len(wrong)*100:.1f}%), "
              f"both directions")
        for k, v in tally.items():
            print(f"    {k:28} {v:5,}  ({v/max(len(sel),1)*100:5.1f}%)")
        for a, b in ((A, B), (B, A)):
            n = ((sel["true_label"].str.strip() == a)
                 & (sel["predicted_label"].str.strip() == b)).sum()
            print(f"    {a} -> {b}: {n:,}")
        other = wrong[~wrong.index.isin(sel.index)]
        print(f"  mean confidence on the pair {sel['top1_prob'].mean():.3f} "
              f"vs {other['top1_prob'].mean():.3f} on its other errors")
        if tally["names the PREDICTED label"] > tally["names the TRUE label"]:
            print(f"  >> on most of this pair the model follows {SOURCE_FIELD} and the")
            print(f"     label disagrees with it. That is a taxonomy question for the")
            print(f"     sponsor, not something training fixes.")
        print()

## 9. Summary

In [ ]:
out = DATA/"interim"; out.mkdir(parents=True, exist_ok=True)
summary = SCORES.copy()
summary["target"] = TARGET
for name in summary["run"]:
    if name in CAL:
        summary.loc[summary["run"] == name, "ECE_%"] = round(CAL[name]["ece"]*100, 2)
        summary.loc[summary["run"] == name, "ECE_scaled_%"] = round(CAL[name]["ece_scaled"]*100, 2)
        summary.loc[summary["run"] == name, "temperature"] = round(CAL[name]["T"], 2)
        summary.loc[summary["run"] == name, "brier"] = round(CAL[name]["brier"], 4)
print(summary.to_string(index=False))
summary.to_csv(out/f"evaluation_summary_{TARGET.lower()}.csv", index=False)
print("\nwritten to", (out/f"evaluation_summary_{TARGET.lower()}.csv").relative_to(ROOT))
print("(data/interim/ is gitignored, so this file is not in the repo)")

print()
print("Guarantees this notebook enforces rather than assumes:")
print(f"  reserved test rows scored                  : 0 (loader refuses them)")
print(f"  run tied to the split it names             : "
      f"{sum(1 for r in RUNS.values() if r.get('split_checked'))} of {len(RUNS)} runs carry")
print(f"                                               validation_record_ids_sha256; those")
print(f"                                               must match this split or be refused")
print(f"  probability columns mapped to classes      : read from the run's own manifest,")
print(f"                                               then checked against the argmax")
print(f"                                               of the probabilities, or rejected")
print(f"  bootstrap resampled over product groups    : yes, not over rows")
print(f"  temperature scaling leaves accuracy alone  : asserted in section 4 (real assert)")
print()
print("What is NOT answered here, and should not be read as answered:")
print("  - no model is selected. that is a team decision with cost in it.")
print("  - review rule 5 (relationship violations) is unwired, so unmeasured.")
print("  - cost per run must come from the training manifests, not from predictions.")
print(f"  - only {TARGET}. set TARGET in section 0 for another, once predictions exist.")